In [71]:
import json
import pandas as pd
import numpy as np

json_file_path = 'captures/migration_decrypted_export.json'



In [72]:
def as_list_on_duplicate_keys(ordered_pairs):
    """
    A custom JSON object_pairs_hook that collects values for duplicate
    keys into a list.
    """
    d = {}
    for k, v in ordered_pairs:
        if k in d:
            if isinstance(d[k], list):
                d[k].append(v)
            else:
                d[k] = [d[k], v]
        else:
            d[k] = v
    return d

In [73]:
try:
    with open(json_file_path, 'r') as f:
        pcap_data = json.load(f, object_pairs_hook=as_list_on_duplicate_keys)
    print(f"Successfully loaded {len(pcap_data)} packets from {json_file_path}")
except FileNotFoundError:
    print(f"Error: The file '{json_file_path}' was not found.")
    pcap_data = []





Successfully loaded 26 packets from captures/migration_decrypted_export.json


In [74]:
# === Helper function to safely get nested fields ===
def get(d, path, default=None):
    """Safely get nested dictionary values using dot notation."""
    for p in path.split('.'):
        if isinstance(d, dict) and p in d:
            d = d[p]
        else:
            return default
    return d

In [75]:
def extract_packet_features(packets):
    """
    Extracts low-level features from a list of packet data.
    Handles multiple QUIC sections properly.
    """
    all_packets_features = []

    if not packets:
        return []

    # Determine server IP from the first two packets to establish direction
    # Assumes the server is the destination of the first packet sent by the client.
    try:
        server_ip = packets[0]['_source']['layers']['ip']['ip.dst']
        print(f"Server IP identified as: {server_ip}")
    except (KeyError, IndexError):
        print("Could not determine server IP automatically. Please check the JSON data.")
        return []

    for packet_data in packets:
        try:
            layers = packet_data['_source']['layers']
            frame_info = layers.get('frame', {})
            ip_info = layers.get('ip', {})
            
            # Handle QUIC layers - could be single dict or list of dicts
            quic_raw = layers.get('quic')
            if quic_raw is None:
                # No QUIC layer, skip this packet
                continue
                
            # Normalize to list format
            if isinstance(quic_raw, list):
                quic_sections = quic_raw
            else:
                quic_sections = [quic_raw]
            
            # Collect frames from all sections
            all_frames = []
            for quic_section in quic_sections:
                frames = quic_section.get('quic.frame', [])
                if not isinstance(frames, list):
                    frames = [frames] if frames else []
                all_frames.extend(frames)
            
            # --- Basic Features ---
            delta_time = float(frame_info.get('frame.time_delta', 0.0))
            packet_length = int(frame_info.get('frame.len', 0))
            frame_number = int(frame_info.get('frame.number', 0))
            packet_direction = 1 if ip_info.get('ip.src') == server_ip else 0

            # --- QUIC Header Features ---
            # Check if ANY section has a long header
            has_any_long_header = any('quic.long.packet_type' in section for section in quic_sections)
            header_form = 1 if has_any_long_header else 0
            # --- QUIC Packet Type (One-Hot Encoded) ---
            # Check ALL sections for packet types - a packet can contain multiple types
            is_initial, is_0rtt, is_handshake, is_retry, is_vn, is_1rtt = 0, 0, 0, 0, 0, 0
            
            # Check each QUIC section for packet types
            for quic_section in quic_sections:
                # Check for version negotiation
                if quic_section.get('quic.version') == '0x00000000':
                    is_vn = 1
                
                # Check for long header packet types
                if 'quic.long.packet_type' in quic_section:
                    pkt_type = quic_section.get('quic.long.packet_type')
                    if pkt_type == '0': 
                        is_initial = 1
                    elif pkt_type == '1': 
                        is_0rtt = 1
                    elif pkt_type == '2': 
                        is_handshake = 1
                    elif pkt_type == '3': 
                        is_retry = 1
                else:
                    # Short header = 1-RTT packet (check if this section has short header indicators)
                    if 'quic.short' in quic_section or not has_any_long_header:
                        is_1rtt = 1

            # --- QUIC Frame-based Features ---
            has_path_challenge, has_path_response, has_new_cid, has_retire_cid = 0, 0, 0, 0
            has_padding, has_ack, has_close = 0, 0, 0
            
            http3_stream_ids = set()
            http3_fin_count = 0

            # Process all frames from all QUIC sections
            for frame in all_frames:
                try:
                    frame_type_str = frame.get('quic.frame_type', '0x-1')
                    if frame_type_str.startswith('0x'):
                        frame_type = int(frame_type_str, 16)
                    else:
                        frame_type = int(frame_type_str)

                    if frame_type == 0x00: 
                        has_padding = 1
                    elif frame_type in [0x02, 0x03]: 
                        has_ack = 1
                    elif frame_type in [0x1c, 0x1d]: 
                        has_close = 1
                    elif frame_type == 0x18: 
                        has_new_cid = 1
                    elif frame_type == 0x19: 
                        has_retire_cid = 1
                    elif frame_type == 0x1a: 
                        has_path_challenge = 1
                    elif frame_type == 0x1b: 
                        has_path_response = 1
                    
                    # Stream related features
                    if 0x08 <= frame_type <= 0x0f:
                        stream_id = frame.get('quic.stream.stream_id')
                        if stream_id:
                            http3_stream_ids.add(stream_id)
                        
                        # Check for FIN bit
                        fin_bit = frame.get('quic.frame_type_tree', {}).get('quic.stream.fin', '0')
                        if fin_bit == '1':
                            http3_fin_count += 1
                            
                except (ValueError, TypeError) as e:
                    # Skip frames with invalid frame types
                    continue
            
            http3_stream_count = len(http3_stream_ids)
            
            # Append all extracted features for this packet
            features = {
                'delta_time': delta_time,
                'packet_length': packet_length,
                'frame_number': frame_number,
                'packet_direction': packet_direction,
                'header_form': header_form,
                'is_initial': is_initial,
                'is_0rtt': is_0rtt,
                'is_handshake': is_handshake,
                'is_1rtt': is_1rtt,
                'is_retry': is_retry,
                'is_vn': is_vn,
                'has_path_challenge': has_path_challenge,
                'has_path_response': has_path_response,
                'has_new_connection_id': has_new_cid,
                'has_retire_cid': has_retire_cid,
                'has_padding': has_padding,
                'has_ack': has_ack,
                'has_connection_close': has_close,
                'http3_stream_count': http3_stream_count,
                'http3_fin_count': http3_fin_count
            }
            all_packets_features.append(features)

        except Exception as e:
            packet_num = frame_info.get('frame.number', 'N/A')
            print(f"Could not process packet {packet_num}. Error: {e}. Skipping...")

    return all_packets_features

# Run the extraction function
extracted_features = extract_packet_features(pcap_data)
print(f"\nFeature extraction complete. Extracted features for {len(extracted_features)} packets.")

Server IP identified as: 127.0.0.1

Feature extraction complete. Extracted features for 26 packets.


In [76]:
if extracted_features:
    # Define the desired order of columns for the CSV file
    column_order = [
        'frame_number', 'delta_time', 'packet_length', 'packet_direction', 'header_form',
        'is_initial', 'is_0rtt', 'is_handshake', 'is_1rtt', 'is_retry', 'is_vn',
        'has_ack', 'has_padding', 'has_connection_close',
        'has_path_challenge', 'has_path_response', 
        'has_new_connection_id', 'has_retire_cid',
        'http3_stream_count', 'http3_fin_count'
    ]

    # Create a pandas DataFrame from the list of feature dictionaries
    features_df = pd.DataFrame(extracted_features)
    
    # Only reorder columns that exist in the DataFrame
    available_columns = [col for col in column_order if col in features_df.columns]
    features_df = features_df[available_columns]

    # Save the DataFrame to a CSV file
    csv_output_path = 'low_level_features.csv'
    features_df.to_csv(csv_output_path, index=False)

    print(f"\nDataFrame created successfully and saved to '{csv_output_path}'")
    
    # Display the first 10 rows of the created DataFrame for verification
    display(features_df)
else:
    print("\nNo features were extracted. CSV file not created.")


DataFrame created successfully and saved to 'low_level_features.csv'


,frame_number,delta_time,packet_length,packet_direction,header_form,is_initial,is_0rtt,is_handshake,is_1rtt,is_retry,is_vn,has_ack,has_padding,has_connection_close,has_path_challenge,has_path_response,has_new_connection_id,has_retire_cid,http3_stream_count,http3_fin_count
0,371,0.051386,1232,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
1,372,0.000216,79,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0
2,373,0.000491,1232,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,374,0.000160,121,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
4,375,0.000460,1232,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,376,0.002193,1232,1,1,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0
6,377,0.000113,501,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
7,378,0.001844,1382,0,1,1,0,1,1,0,0,1,1,0,0,0,1,0,0,0
8,379,0.000774,540,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,1,0
9,380,0.000115,76,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0
